In [ ]:
import pandas as pd  # Importamos pandas para trabajar las con tablas.

# Leemos el CSV desde la carpeta de datos del proyecto para que pueda ser leido mas facil desde otro dispotivo.
# ../ nos permite salir de notebooks y entrar a data/raw.
ventas = pd.read_csv("../data/raw/ventas_2009_2011.csv", encoding="latin-1")

# Mostramos las primeras cinco filas para conocer la información.
display(ventas.head())

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
# Cargamos el segundo archivo para comparar ambas fuentes
ventas_recientes = pd.read_csv(
    "../data/raw/ventas_2010_2011.csv",
    encoding="latin-1"
)

# Revisamos cada archivo por separado
for nombre, datos in [
    ("Ventas históricas", ventas),
    ("Ventas recientes", ventas_recientes)
]:
    print(f"\n- {nombre} -")

    # Conocemos la cantidad de filas y columnas
    print("Dimensiones:", datos.shape)

    # Identificamos cómo se llaman las columnas.
    print("Columnas:", datos.columns.tolist())

    # Contamos los datos faltantes por columna
    print("\nDatos faltantes:")
    print(datos.isnull().sum())

    # Identificamos filas exactamente repetidas.
    print("\nFilas duplicadas:", datos.duplicated().sum())


- Ventas históricas -
Dimensiones: (1067371, 8)
Columnas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Datos faltantes:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Filas duplicadas: 34335

- Ventas recientes -
Dimensiones: (541909, 8)
Columnas: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Datos faltantes:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Filas duplicadas: 5268


### comprobamos si existe superposición en las fuentes

In [6]:
# Creamos una copia para no modificar los datos originales.
historicas = ventas.copy()
recientes = ventas_recientes.copy()

# Igualamos los nombres de las columnas para poder comparar.
historicas = historicas.rename(columns={
    "Invoice": "factura",
    "StockCode": "producto",
    "Description": "descripcion",
    "Quantity": "cantidad",
    "InvoiceDate": "fecha",
    "Price": "precio",
    "Customer ID": "cliente",
    "Country": "pais"
})

recientes = recientes.rename(columns={
    "InvoiceNo": "factura",
    "StockCode": "producto",
    "Description": "descripcion",
    "Quantity": "cantidad",
    "InvoiceDate": "fecha",
    "UnitPrice": "precio",
    "CustomerID": "cliente",
    "Country": "pais"
})

# Contamos las facturas que aparecen en ambos archivos.
facturas_historicas = set(historicas["factura"].astype(str))
facturas_recientes = set(recientes["factura"].astype(str))

facturas_compartidas = facturas_historicas.intersection(facturas_recientes)

print("Facturas compartidas:", len(facturas_compartidas))
print("Facturas únicas del archivo histórico:", len(facturas_historicas))
print("Facturas únicas del archivo reciente:", len(facturas_recientes))

Facturas compartidas: 25900
Facturas únicas del archivo histórico: 53628
Facturas únicas del archivo reciente: 25900


### Verificamos si las transacciones recientes ya están en el histórico

In [7]:
# Elegimos las columnas que identifican el contenido de una venta.
columnas = [
    "factura",
    "producto",
    "descripcion",
    "cantidad",
    "fecha",
    "precio",
    "cliente",
    "pais"
]

# Creamos copias para comparar sin modificar los datos originales.
datos_historicos = historicas[columnas].copy()
datos_recientes = recientes[columnas].copy()

# Convertimos los valores a texto y quitamos espacios sobrantes.
# Así evitamos diferencias causadas solo por el formato.
for datos in [datos_historicos, datos_recientes]:
    for columna in columnas:
        datos[columna] = datos[columna].astype("string").str.strip()

# Contamos cuántas veces aparece cada fila en el histórico.
# Esto permite considerar también las filas repetidas.
conteo_historico = (
    datos_historicos
    .value_counts(subset=columnas, dropna=False)
    .rename("veces_historico")
    .reset_index()
)

# Contamos cuántas veces aparece cada fila en el archivo reciente.
conteo_reciente = (
    datos_recientes
    .value_counts(subset=columnas, dropna=False)
    .rename("veces_reciente")
    .reset_index()
)

# Comparamos los registros de ambos archivos.
comparacion = conteo_reciente.merge(
    conteo_historico,
    on=columnas,
    how="left"
)

# Cuando una fila no existe en el histórico, su conteo será cero.
comparacion["veces_historico"] = (
    comparacion["veces_historico"].fillna(0)
)

# Calculamos cuántas filas recientes no están cubiertas
# por las mismas filas del archivo histórico.
filas_no_cubiertas = (
    comparacion["veces_reciente"] - comparacion["veces_historico"]
).clip(lower=0).sum()

print("Filas del archivo reciente:", len(datos_recientes))
print("Filas recientes no cubiertas por el histórico:", int(filas_no_cubiertas))

Filas del archivo reciente: 541909
Filas recientes no cubiertas por el histórico: 541909


In [8]:
# Seleccionamos tres columnas básicas de cada venta.
columnas_clave = ["factura", "producto", "cantidad"]

# Preparamos una copia de cada archivo para compararlos.
historico_claves = historicas[columnas_clave].copy()
reciente_claves = recientes[columnas_clave].copy()

# Igualamos el formato de las columnas.
for datos in [historico_claves, reciente_claves]:

    # Convertimos factura y producto a texto y quitamos espacios.
    datos["factura"] = datos["factura"].astype(str).str.strip()
    datos["producto"] = datos["producto"].astype(str).str.strip()

    # Convertimos cantidad a número para evitar diferencias de formato.
    datos["cantidad"] = pd.to_numeric(datos["cantidad"], errors="coerce")

# Obtenemos las combinaciones diferentes de factura, producto y cantidad.
claves_historicas = historico_claves.drop_duplicates()
claves_recientes = reciente_claves.drop_duplicates()

# Buscamos cuáles combinaciones del archivo reciente están en el histórico.
comparacion_claves = claves_recientes.merge(
    claves_historicas,
    on=columnas_clave,
    how="left",
    indicator=True
)

# Contamos las combinaciones encontradas y las no encontradas.
print("Resultado de la comparación:")
print(comparacion_claves["_merge"].value_counts())

# Calculamos qué porcentaje de combinaciones recientes aparece en el histórico.
porcentaje = (
    (comparacion_claves["_merge"] == "both").mean() * 100
)

print(f"\nCoincidencia entre fuentes: {porcentaje:.2f}%")

Resultado de la comparación:
_merge
both          536478
left_only          0
right_only         0
Name: count, dtype: int64

Coincidencia entre fuentes: 100.00%


prueba de transformacion Etl transformacion_ventas


In [9]:
import sys  # Permite indicar dónde buscar nuestros archivos Python.

# Agregamos la carpeta principal del proyecto.
sys.path.insert(0, "..")

# Importamos la función que creamos en el archivo ETL.
from etl.transformacion_ventas import transformar_ventas

# Transformamos una pequeña muestra para verificar el resultado.
muestra = transformar_ventas(ventas.head(5))

# Mostramos las columnas transformadas.
display(muestra)

,factura,producto,descripcion,cantidad,fecha,precio,cliente,pais,tipo_movimiento,total
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00+00:00,6.95,13085.0,United Kingdom,venta,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00+00:00,6.75,13085.0,United Kingdom,venta,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00+00:00,6.75,13085.0,United Kingdom,venta,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00+00:00,2.10,13085.0,United Kingdom,venta,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00+00:00,1.25,13085.0,United Kingdom,venta,30.0


In [10]:
# Importamos la función que revisa la calidad de las ventas.
from etl.validacion_ventas import validar_ventas

# Aplicamos la transformación a todas las ventas históricas.
ventas_transformadas = transformar_ventas(ventas)

# Separamos las ventas que cumplen las reglas de las rechazadas.
ventas_validas, ventas_rechazadas = validar_ventas(ventas_transformadas)

# Contamos los registros de cada grupo.
print("Registros originales:", len(ventas))
print("Registros válidos:", len(ventas_validas))
print("Registros rechazados:", len(ventas_rechazadas))

# Verificamos que ningún registro se haya perdido durante la separación.
print(
    "Total conservado:",
    len(ventas_validas) + len(ventas_rechazadas) == len(ventas)
)

Registros originales: 1067371
Registros válidos: 1064621
Registros rechazados: 2750
Total conservado: True


In [11]:
# Contamos los registros rechazados por cada problema.

# Revisamos cantidades iguales a cero.
cantidad_cero = (ventas_rechazadas["cantidad"] == 0).sum()

# Revisamos precios iguales a cero.
precio_cero = (ventas_rechazadas["precio"] == 0).sum()

# Revisamos precios negativos.
precio_negativo = (ventas_rechazadas["precio"] < 0).sum()

# Revisamos fechas que no pudieron convertirse.
fecha_invalida = ventas_rechazadas["fecha"].isna().sum()

# Mostramos los resultados para decidir qué hacer.
print("Cantidad igual a cero:", cantidad_cero)
print("Precio igual a cero:", precio_cero)
print("Precio negativo:", precio_negativo)
print("Fecha inválida:", fecha_invalida)

# Mostramos cinco registros rechazados como ejemplo.
display(ventas_rechazadas.head())

Cantidad igual a cero: 0
Precio igual a cero: 2745
Precio negativo: 5
Fecha inválida: 0


,factura,producto,descripcion,cantidad,fecha,precio,cliente,pais,tipo_movimiento,total,motivo
3161,489659,21350,NaN,230,2009-12-01 17:39:00+00:00,0.0,NaN,United Kingdom,venta,0.0,Registro con datos obligatorios inválidos
3731,489781,84292,NaN,17,2009-12-02 11:45:00+00:00,0.0,NaN,United Kingdom,venta,0.0,Registro con datos obligatorios inválidos
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00+00:00,0.0,16126.0,United Kingdom,venta,0.0,Registro con datos obligatorios inválidos
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00+00:00,0.0,NaN,United Kingdom,venta,0.0,Registro con datos obligatorios inválidos
6378,489882,35751C,NaN,12,2009-12-02 16:22:00+00:00,0.0,NaN,United Kingdom,venta,0.0,Registro con datos obligatorios inválidos
